# Diabetic Retinopathy Classification — Clean Colab GPU Notebook

A fresh, linear notebook for training EfficientNet-B0 on APTOS with GPU support, Drive checkpoints, resume support, and final test evaluation.

**Expected Google Drive folder:** `MyDrive/DR_screening_project/`
```text
DR_screening_project/
├── data/train_images/
├── data/splits/train.csv
├── data/splits/val.csv
├── data/splits/test.csv
└── models/   (created automatically)
```
In Colab, select **Runtime → Change runtime type → T4 GPU** if available, then run cells top-to-bottom. Checkpoints and logs are saved to Drive. This is an educational prototype, not a clinically validated diagnostic tool.


In [ ]:
# 1. Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/DR_screening_project')
DATA_DIR = PROJECT_DIR / 'data'
IMAGE_DIR = DATA_DIR / 'train_images'
SPLIT_DIR = DATA_DIR / 'splits'
MODEL_DIR = PROJECT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for p in [PROJECT_DIR, IMAGE_DIR, SPLIT_DIR/'train.csv', SPLIT_DIR/'val.csv', SPLIT_DIR/'test.csv']:
    print(('OK      ' if p.exists() else 'MISSING '), p)


In [ ]:
# 2. Dependencies and device check
!pip -q install -U scikit-learn pandas pillow

import os, time, json, random
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import cohen_kappa_score, accuracy_score, classification_report, confusion_matrix

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: Enable a GPU in Runtime > Change runtime type.')


In [ ]:
# 3. Configuration
SEED = 42
IMAGE_SIZE = 256
BATCH_SIZE = 32 if torch.cuda.is_available() else 8
EPOCHS = 10
NUM_CLASSES = 5
NUM_WORKERS = 2
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BEST_PATH = MODEL_DIR / 'best.pt'
LATEST_PATH = MODEL_DIR / 'latest.pt'
LOG_PATH = MODEL_DIR / 'training_log.csv'

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()
print('Device:', DEVICE, '| batch:', BATCH_SIZE, '| epochs:', EPOCHS)


In [ ]:
# 4. Load and validate splits
required = [PROJECT_DIR, IMAGE_DIR, SPLIT_DIR/'train.csv', SPLIT_DIR/'val.csv', SPLIT_DIR/'test.csv']
missing_paths = [str(p) for p in required if not p.exists()]
if missing_paths:
    raise FileNotFoundError('Missing paths:\\n' + '\\n'.join(missing_paths) +
        '\\nCheck that your folder is MyDrive/DR_screening_project and splits are in data/splits/.')

def normalize_columns(df):
    df = df.copy()
    lower = {c.lower().strip(): c for c in df.columns}
    id_col = next((lower[k] for k in ['id_code','image_id','image','filename','file_name'] if k in lower), None)
    label_col = next((lower[k] for k in ['diagnosis','label','target','class'] if k in lower), None)
    if id_col is None or label_col is None:
        raise ValueError(f'Expected image ID and diagnosis columns; found {df.columns.tolist()}')
    df = df.rename(columns={id_col:'id_code', label_col:'diagnosis'})
    df['id_code'] = df['id_code'].astype(str).str.strip()
    df['diagnosis'] = pd.to_numeric(df['diagnosis'], errors='raise').astype(int)
    return df[['id_code','diagnosis']]

train_df = normalize_columns(pd.read_csv(SPLIT_DIR/'train.csv'))
val_df = normalize_columns(pd.read_csv(SPLIT_DIR/'val.csv'))
test_df = normalize_columns(pd.read_csv(SPLIT_DIR/'test.csv'))
print('Rows — train:', len(train_df), 'validation:', len(val_df), 'test:', len(test_df))
display(train_df.head())


In [ ]:
# 5. Resolve image files and check split integrity
def resolve_image_path(image_id):
    name = str(image_id)
    candidates = [IMAGE_DIR/name]
    if Path(name).suffix == '':
        candidates += [IMAGE_DIR/(name + ext) for ext in ['.png','.jpg','.jpeg']]
    for p in candidates:
        if p.exists(): return p
    return None

for split_name, df in [('train',train_df),('val',val_df),('test',test_df)]:
    missing = [x for x in df.id_code if resolve_image_path(x) is None]
    print(f'{split_name}: {len(df)} rows; missing images: {len(missing)}; labels:', df.diagnosis.value_counts().sort_index().to_dict())
    if missing:
        raise FileNotFoundError(f'Missing {split_name} images, examples: {missing[:5]}')


In [ ]:
# 6. Dataset and data loaders
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.12, contrast=0.12),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)
])
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)
])

class RetinopathyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with Image.open(resolve_image_path(row.id_code)) as im:
            image = im.convert('RGB')
        return self.transform(image), int(row.diagnosis)

loader_args = {'num_workers': NUM_WORKERS, 'pin_memory': torch.cuda.is_available()}
train_loader = DataLoader(RetinopathyDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, **loader_args)
val_loader = DataLoader(RetinopathyDataset(val_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, **loader_args)
test_loader = DataLoader(RetinopathyDataset(test_df, eval_transform), batch_size=BATCH_SIZE, shuffle=False, **loader_args)
xb, yb = next(iter(train_loader))
print('Sample batch:', tuple(xb.shape), tuple(yb.shape))


In [ ]:
# 7. Build EfficientNet-B0
try:
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    print('Loaded ImageNet pretrained weights.')
except Exception as e:
    print('Could not load pretrained weights; using random initialization:', repr(e))
    model = models.efficientnet_b0(weights=None)

model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
print('Model ready.')


In [ ]:
# 8. Training helpers
def run_epoch(loader, training):
    model.train(training)
    total_loss, total_n = 0.0, 0
    ys, preds_all = [], []
    with (torch.enable_grad() if training else torch.no_grad()):
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            pred = logits.argmax(1)
            n = labels.size(0)
            total_loss += loss.item()*n; total_n += n
            ys.extend(labels.detach().cpu().tolist())
            preds_all.extend(pred.detach().cpu().tolist())
    return (total_loss/max(total_n,1), accuracy_score(ys,preds_all),
            cohen_kappa_score(ys,preds_all,weights='quadratic'), ys, preds_all)

def save_checkpoint(path, epoch, best_qwk):
    torch.save({'epoch':epoch, 'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(),
                'scheduler_state_dict':scheduler.state_dict(), 'best_qwk':best_qwk,
                'architecture':'efficientnet_b0', 'image_size':IMAGE_SIZE}, path)


In [ ]:
# 9. TRAIN — saves latest checkpoint each epoch and best checkpoint by validation QWK
# If starting fresh, run this cell. If resuming after disconnect, run the resume cell first.
start_epoch = 0
best_qwk = -float('inf')
history = []

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc, tr_qwk, _, _ = run_epoch(train_loader, training=True)
    va_loss, va_acc, va_qwk, _, _ = run_epoch(val_loader, training=False)
    scheduler.step(va_qwk)
    row = {'epoch':epoch+1, 'train_loss':tr_loss, 'train_accuracy':tr_acc, 'train_qwk':tr_qwk,
           'val_loss':va_loss, 'val_accuracy':va_acc, 'val_qwk':va_qwk,
           'lr':optimizer.param_groups[0]['lr'], 'seconds':time.time()-t0}
    history.append(row)
    pd.DataFrame(history).to_csv(LOG_PATH, index=False)
    save_checkpoint(LATEST_PATH, epoch+1, max(best_qwk, va_qwk))
    if va_qwk > best_qwk:
        best_qwk = va_qwk
        save_checkpoint(BEST_PATH, epoch+1, best_qwk)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss={tr_loss:.4f} acc={tr_acc:.4f} | val loss={va_loss:.4f} acc={va_acc:.4f} QWK={va_qwk:.4f} | {row['seconds']:.0f}s | best QWK={best_qwk:.4f}")
    print('Drive checkpoint saved.')
print('Training finished. Best checkpoint:', BEST_PATH)


## Resume after a Colab disconnect

If Colab disconnects, re-open this notebook and run cells 1–8 to mount Drive and recreate the model/loaders. **Do not run the training cell yet.** Run the next cell to restore `latest.pt`, then run the training cell. It will continue from the saved epoch. If the saved epoch is already 10, increase `EPOCHS` in cell 3 first.


In [ ]:
# 10. OPTIONAL RESUME — use only after a disconnect, before training
if LATEST_PATH.exists():
    ckpt = torch.load(LATEST_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = int(ckpt['epoch'])
    best_qwk = float(ckpt.get('best_qwk', -float('inf')))
    if LOG_PATH.exists(): history = pd.read_csv(LOG_PATH).to_dict('records')
    print(f'Restored epoch {start_epoch}; best QWK so far {best_qwk:.4f}. Run the training cell next.')
else:
    print('No latest checkpoint found. Start fresh with the training cell.')


In [ ]:
# 11. Final evaluation — held-out test split, using best validation checkpoint
if not BEST_PATH.exists():
    raise FileNotFoundError('best.pt not found. Train the model first.')
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
test_loss, test_acc, test_qwk, y_true, y_pred = run_epoch(test_loader, training=False)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test quadratic weighted kappa: {test_qwk:.4f}')
print('\\nClassification report:')
print(classification_report(y_true, y_pred, labels=[0,1,2,3,4], zero_division=0))
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred, labels=[0,1,2,3,4]))
summary = {'best_epoch':int(ckpt.get('epoch',-1)), 'best_validation_qwk':float(ckpt.get('best_qwk',float('nan'))),
           'test_loss':float(test_loss), 'test_accuracy':float(test_acc), 'test_qwk':float(test_qwk),
           'device':str(DEVICE), 'architecture':'EfficientNet-B0'}
with open(MODEL_DIR/'evaluation_summary.json','w') as f: json.dump(summary,f,indent=2)
print('Saved summary to', MODEL_DIR/'evaluation_summary.json')
